In [ ]:
!pip install pycocotools

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pycocotools.coco import COCO
import skimage.io as io
from PIL import Image
from PIL import ImageDraw

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_dir = '/content/drive/My Drive/Car-Parts-Segmentation-master/Car-Parts-Segmentation-master/trainingset/annotations.json'
coco = COCO(data_dir)

loading annotations into memory...
Done (t=0.37s)
creating index...
index created!


In [ ]:
carIDS = coco.getCatIds()
cars = coco.loadCats(carIDS)
car_id = {}
for i, j in zip(carIDS, cars):

  car_id[i] = j['name']
car_id

{1: '_background_',
 2: 'back_bumper',
 3: 'back_glass',
 4: 'back_left_door',
 5: 'back_left_light',
 6: 'back_right_door',
 7: 'back_right_light',
 8: 'front_bumper',
 9: 'front_glass',
 10: 'front_left_door',
 11: 'front_left_light',
 12: 'front_right_door',
 13: 'front_right_light',
 14: 'hood',
 15: 'left_mirror',
 16: 'right_mirror',
 17: 'tailgate',
 18: 'trunk',
 19: 'wheel'}

In [ ]:
print('ID \t: Class Name')
print("-------")
for i in cars:
  print('{}\t: {}'.format(i['id'], i['name']))

ID 	: Class Name
-------
1	: _background_
2	: back_bumper
3	: back_glass
4	: back_left_door
5	: back_left_light
6	: back_right_door
7	: back_right_light
8	: front_bumper
9	: front_glass
10	: front_left_door
11	: front_left_light
12	: front_right_door
13	: front_right_light
14	: hood
15	: left_mirror
16	: right_mirror
17	: tailgate
18	: trunk
19	: wheel


In [ ]:
img_ids = coco.getImgIds()
images = coco.loadImgs(img_ids)

In [ ]:
images

[{'id': 1,
  'dataset_id': 1,
  'category_ids': [],
  'path': 'JPEGImages/train1.jpg',
  'width': 512,
  'height': 512,
  'file_name': 'train1.jpg',
  'annotated': False,
  'annotating': [],
  'num_annotations': 0,
  'metadata': {},
  'deleted': False,
  'milliseconds': 0,
  'events': [],
  'regenerate_thumbnail': False},
 {'id': 2,
  'dataset_id': 1,
  'category_ids': [],
  'path': 'JPEGImages/train10.jpg',
  'width': 512,
  'height': 512,
  'file_name': 'train10.jpg',
  'annotated': False,
  'annotating': [],
  'num_annotations': 0,
  'metadata': {},
  'deleted': False,
  'milliseconds': 0,
  'events': [],
  'regenerate_thumbnail': False},
 {'id': 3,
  'dataset_id': 1,
  'category_ids': [],
  'path': 'JPEGImages/train100.jpg',
  'width': 512,
  'height': 512,
  'file_name': 'train100.jpg',
  'annotated': False,
  'annotating': [],
  'num_annotations': 0,
  'metadata': {},
  'deleted': False,
  'milliseconds': 0,
  'events': [],
  'regenerate_thumbnail': False},
 {'id': 4,
  'dataset_

In [ ]:
annIds = coco.getAnnIds(img_ids)
annImgs = coco.loadAnns(annIds)
annImgs

[{'id': 111,
  'image_id': 1,
  'category_id': 19,
  'segmentation': [[132.1,
    345.3,
    141.5,
    352.4,
    149.2,
    366.5,
    152.1,
    381.8,
    150.4,
    397.6,
    144.5,
    411.2,
    135.6,
    420.0,
    124.5,
    422.9,
    110.9,
    416.5,
    102.7,
    401.2,
    98.6,
    386.5,
    100.9,
    370.6,
    105.1,
    358.2,
    112.7,
    347.6,
    120.4,
    343.5],
   [391.5,
    346.5,
    400.4,
    354.7,
    406.2,
    367.6,
    409.8,
    382.9,
    406.8,
    400.0,
    401.5,
    412.4,
    393.3,
    421.2,
    382.7,
    423.5,
    374.5,
    421.8,
    365.1,
    414.7,
    358.0,
    401.2,
    355.6,
    385.3,
    358.6,
    367.6,
    363.3,
    354.1,
    372.7,
    346.5,
    380.9,
    343.5]],
  'area': 6479,
  'bbox': [99.0, 344.0, 311.0, 80.0],
  'iscrowd': False,
  'isbbox': False,
  'color': '#e45779',
  'metadata': {}},
 {'id': 112,
  'image_id': 1,
  'category_id': 10,
  'segmentation': [[309.8,
    247.1,
    303.9,
    289.4,
    

In [ ]:
import os
imgPath = "/content/drive/My Drive/Car-Parts-Segmentation-master/Car-Parts-Segmentation-master/trainingset/JPEGImages"
print(os.path.exists(imgPath))

True


In [ ]:
image_data = []
image_annotated_data = []
category_ids = []
imgPath = "/content/drive/My Drive/Car-Parts-Segmentation-master/Car-Parts-Segmentation-master/trainingset/JPEGImages"

txt_path = "/content/drive/MyDrive/Car-Parts-Segmentation-master/Car-Parts-Segmentation-master/annotations.txt"

with open(txt_path, 'w') as txt_file:

  for img in os.listdir(imgPath):

    try:
        img_path = os.path.join(imgPath, img)

        image = Image.open(img_path).convert('RGB')
        img_width, img_height = image.size
        image_data.append({'Img': image.copy()})
        image_id = int(img.split('train')[1].split('.jpg')[0])
        ann_id = coco.getAnnIds(imgIds = image_id)
        annotations = coco.loadAnns(ann_id)


        for ann in annotations:

              if 'bbox' in ann:

                  bbox = ann['bbox']

                  xmin, ymin, xmax, ymax = bbox

              if xmin > xmax:
                  xmin, xmax = xmax, xmin  # Swap to ensure xmin < xmax
              if ymin > ymax:
                  ymin, ymax = ymax, ymin  # Swap to ensure ymin < ymax

                  x_center = (xmin + xmax) / (2*img_width)
                  y_center = (ymin + ymax) / (2*img_height)
                  width = (xmax - xmin)/img_width
                  height = (ymax - ymin)/img_height

                  class_id = ann['category_id']

                  img_id = ann['image_id']

                  if class_id is not None:

                    txt_file.write(f"{img_id} {class_id-1} {x_center} {y_center} {width} {height}\n")















    except FileNotFoundError:
            print(f"Image not found")



In [ ]:
from collections import defaultdict
import os


output_annotations_files = "/content/drive/MyDrive/Car-Parts-Segmentation-master/Car-Parts-Segmentation-master/train/labels"
os.makedirs(output_annotations_files, exist_ok=True)

with open(txt_path, 'r') as file:
    rows = file.readlines()


data = [row.strip().split() for row in rows if row.strip()]  # Skip empty lines


grouped = defaultdict(list)
for row in data:
    try:
        img_id = int(row[0])
        grouped[img_id].append(row[1:])
    except (IndexError, ValueError):
        print(f"Skipping invalid row: {row}")


for img_id, bboxes in sorted(grouped.items()):
    output_file_path = os.path.join(output_annotations_files, f"train{img_id}.txt")

    with open(output_file_path, 'w') as txt_file:
        for bbox in bboxes:
            if len(bbox) == 5:
                txt_file.write(f"{bbox[0]} {bbox[1]} {bbox[2]} {bbox[3]} {bbox[4]}\n")
            else:
                print(f"Skipping invalid bbox for image {img_id}: {bbox}")

print("Annotation files created successfully.")


Annotation files created successfully.


In [ ]:
pip install -U albumentations

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 632.7/632.7 kB 33.3 MB/s eta 0:00:00
  Attempting uninstall: albucore
    Found existing installation: albucore 0.0.19
    Uninstalling albucore-0.0.19:
      Successfully uninstalled albucore-0.0.19
  Attempting uninstall: albumentations
    Found existing installation: albumentations 1.4.20
    Uninstalling albumentations-1.4.20:
      Successfully uninstalled albumentations-1.4.20


In [ ]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 904.3/904.3 kB 38.8 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

# Initialize the YOLO model
model = YOLO('yolov8n.pt')  # Specify the pre-trained model

# Train the model
model.train(
    data='/content/drive/MyDrive/Car-Parts-Segmentation-master/Car-Parts-Segmentation-master/data.yaml',  # Path to data.yaml
    epochs=30,        # Number of training epochs
    imgsz=512,        # Image size

)

Ultralytics 8.3.55 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/drive/MyDrive/Car-Parts-Segmentation-master/Car-Parts-Segmentation-master/data.yaml, epochs=30, time=None, patience=100, batch=16, imgsz=512, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=F

train: Scanning /content/drive/MyDrive/Car-Parts-Segmentation-master/Car-Parts-Segmentation-master/train/labels.cache... 369 images, 0 backgrounds, 0 corrupt: 100%|██████████| 369/369 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))



val: Scanning /content/drive/MyDrive/Car-Parts-Segmentation-master/Car-Parts-Segmentation-master/val/labels.cache... 30 images, 1 backgrounds, 0 corrupt: 100%|██████████| 31/31 [00:00<?, ?it/s]


Plotting labels to runs/detect/train2/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000435, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 512 train, 512 val
Using 2 dataloader workers
Logging results to runs/detect/train2
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all         31        177      0.188     0.0483     0.0219    0.00785



30 epochs completed in 0.078 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 6.2MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.55 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
Model summary (fused): 168 layers, 3,009,353 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  3.93it/s]


                   all         31        177       0.27     0.0471     0.0287     0.0103
           back_bumper         10         10          1          0     0.0522     0.0154
            back_glass          9          9          0          0          0          0
        back_left_door          2          2          0          0          0          0
       back_left_light         12         12     0.0864      0.167     0.0353     0.0166
       back_right_door          2          2          0          0          0          0
      back_right_light         12         12          1          0     0.0165    0.00267
          front_bumper          7          7          1          0    0.00521    0.00123
           front_glass          6          6          0          0          0          0
       front_left_door          4          4          0          0          0          0
      front_left_light         14         14     0.0588     0.0714     0.0497     0.0115
      front_right_doo

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bdab1142d70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    